# Module 6: Event Studies and Pre Trend Testing

*Developed by Yin Zhang, PhD, Assistant Professor, Department of Mathematics and Statistics, Washington State University. Part of the Public Safety Statistics Tutorials, developed for WADEPS through CISER.*

---

The event study plot is the standard credibility exhibit in modern difference
in differences work. This module estimates one, finds the individual
coefficients unreadable, and shows that **the usable output is the joint test
on the pre period coefficients, not the picture.**

It then measures what that test can actually detect.

**About 30 minutes.**

## 1. Setup

In [ ]:
# Where the data lives.
#   On Google Colab this reads straight from GitHub.
#   Running from inside a local clone of the repository also works.
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

GITHUB = "https://raw.githubusercontent.com/YinZhangCISER/Public-Safety-Statistics-Tutorials/main/Data/"
_local = Path("../../../Data")
BASE = f"{_local}/" if _local.exists() else GITHUB

monthly = pd.read_csv(BASE + "agency_monthly.csv")
profile = pd.read_csv(BASE + "agency_profile.csv")

# The five agencies that adopted the de escalation training in July 2023.
TRAINED = ["A001", "A002", "A004", "A007", "A010"]
TRUTH = -12.0                       # the effect built into the data, in percent

f = monthly[monthly["provisional"] == 0].copy()          # drop the unfinished months
f = f[~((f["agency_id"] == "A002") & (f["year_month"] == "2021-06"))]   # documented unrest
f["trained"] = f["agency_id"].isin(TRAINED).astype(int)

# Three periods, not two. The program phased in between July and November 2023.
f["period"] = np.where(f["year_month"] >= "2023-11", "after",
                       np.where(f["year_month"] < "2023-07", "before", "phase"))

NAME = dict(zip(profile["agency_id"], profile["agency_name"]))
COMPARISON = sorted(a for a in f["agency_id"].unique() if a not in TRAINED)


def rate(d):
    """Use of force per 100 arrests, pooled over whatever rows are passed in."""
    return 100 * d["n_uof"].sum() / d["n_arrests"].sum()


def cell_rate(agencies, period):
    return rate(f[f["agency_id"].isin(agencies) & (f["period"] == period)])


print(f"{f['agency_id'].nunique()} agencies, {f['year_month'].nunique()} months")
print(f"trained: {', '.join(NAME[a].split()[0] for a in TRAINED)}")

In [ ]:
import statsmodels.api as sm
import statsmodels.formula.api as smf

KEEP = [a for a in TRAINED if a != "A007"]     # the pre trend violator, Intermediate 8
BASELINE = profile.set_index("agency_id")["pre_program_uof_per_100_arrests"]

d = f[f["agency_id"] != "A007"].copy()
d["lo"] = np.log(d["n_arrests"])
pi = pd.PeriodIndex(d["year_month"], freq="M")
d["yr"] = pi.year.values + (pi.month.values - 1) / 12.0
d["base"] = d["agency_id"].map(BASELINE)

pct = lambda b: 100 * (np.exp(b) - 1)


def fit(data, treated, form=None, outcome="n_uof", offset=None):
    s = data.copy()
    s["settled"] = ((s["agency_id"].isin(treated))
                    & (s["period"] == "after")).astype(float)
    s["phase"] = ((s["agency_id"].isin(treated))
                  & (s["period"] == "phase")).astype(float)
    fo = form or f"{outcome} ~ C(agency_id)+C(year_month)+settled+phase"
    z = smf.glm(fo, s, family=sm.families.Poisson(),
                offset=s["lo"] if offset is None else offset).fit()
    lo, hi = z.conf_int().loc["settled"]
    return pct(z.params["settled"]), pct(lo), pct(hi), z

## 2. The event study

One coefficient per month relative to the program's start, with the month
before it as the reference.

In [ ]:
d["t"] = ((pd.PeriodIndex(d["year_month"], freq="M").year - 2019) * 12
          + pd.PeriodIndex(d["year_month"], freq="M").month - 1)
d["k"] = d["t"] - ((2023 - 2019) * 12 + 6)
d["ek"] = np.clip(d["k"], -8, 10).astype(int)
d["trm"] = d["agency_id"].isin(KEEP)
d.loc[~d["trm"], "ek"] = -99

js = [j for j in range(-8, 11) if j != -1]
terms = " + ".join([f"I(trm&(ek=={j}))" for j in js])
z = smf.glm("n_uof ~ C(agency_id)+C(year_month)+" + terms, d,
            family=sm.families.Poisson(), offset=d["lo"]).fit()

rows = []
for j in js:
    k = [c for c in z.params.index if f"ek == {j}" in c][0]
    lo, hi = z.conf_int().loc[k]
    rows.append({"months since start": j, "estimate": f"{pct(z.params[k]):+.1f}%",
                 "95 percent interval": f"[{pct(lo):+.1f}, {pct(hi):+.1f}]"})
pd.DataFrame(rows).set_index("months since start")

The pre period coefficients swing between roughly 20 percent down and 16
percent up, with intervals thirty to fifty points wide, when the truth for
every one of them is zero.

**Reading a story into this plot is reading noise**, and the story a reader
will construct from a jagged pre period is almost always "the trends were
already diverging".

## 3. The test that is actually informative

Test all the pre period coefficients jointly against zero.

In [ ]:
pre_j = [j for j in js if j < 0]
ks = [[c for c in z.params.index if f"ek == {j}" in c][0] for j in pre_j]
R = np.zeros((len(ks), len(z.params)))
for i, k in enumerate(ks):
    R[i, list(z.params.index).index(k)] = 1
w = z.wald_test(R, scalar=False)
print(f"  joint test that all {len(ks)} pre period coefficients are zero")
print(f"    chi squared {float(w.statistic):.1f} on {len(ks)} degrees of freedom, "
      f"p = {float(w.pvalue):.3f}")
print(f"\n  the largest individual pre period coefficient: "
      f"{max(abs(pct(z.params[k])) for k in ks):.1f}%")

The joint test passes comfortably. The largest single pre period coefficient
is about 20 percent.

**Those two facts together are the point.** A reader who looks at the plot
sees a 20 percent pre period swing and doubts the design. A reader who is
given the joint test sees that a set of coefficients this noisy is exactly
what zero looks like at this sample size.

**Report the joint test with the plot**, always. A plot without it invites an
objection that the data already answers.

## 4. What the test can detect

A passing test means nothing until you know what it could have caught.

In [ ]:
rng = np.random.default_rng(4)
pre = d[d["period"] == "before"].copy()


def detect(extra_pct, reps=200):
    hits = 0
    for _ in range(reps):
        s = pre.copy()
        s["tr"] = s["agency_id"].isin(KEEP).astype(float)
        mu = s["n_uof"] * np.exp(np.log(1 + extra_pct / 100) * s["tr"]
                                 * (s["yr"] - s["yr"].min()))
        s["y"] = rng.poisson(np.maximum(mu, 0.01))
        zz = smf.glm("y ~ C(agency_id) + yr + tr:yr", s,
                     family=sm.families.Poisson(), offset=s["lo"]).fit()
        k = [x for x in zz.params.index if "yr" in x and "tr" in x][0]
        hits += zz.pvalues[k] < 0.05
    return 100 * hits / reps


for extra in [1, 2, 3, 5]:
    print(f"  a planted violation of {extra} percent a year is found "
          f"{detect(extra):3.0f} percent of the time")

The violation that mattered in this dataset, the one Summit County creates at
the group level, is **2.16 percent a year**, and a violation that size is
found **16 percent of the time**.

**A passing pre trend test on eleven agencies is weak evidence.** The honest
report gives the test, the interval on the trend difference, and the size of
violation the test had a fair chance of detecting.

## 5. The event study's other uses

| Use | Worth it |
|---|---|
| Testing the pre period jointly | yes, this is the main one |
| Showing the shape of the response | only with far more units than eleven |
| Detecting anticipation at a specific month | yes, if the month is named in advance |
| Reassuring a reader visually | it often does the opposite |

**Recent work has moved toward reporting the pre period test and a sensitivity
bound rather than the plot**, for exactly the reason above. Rambachan and Roth
is the standard reference, and [Module 14](Module_14_Sensitivity_And_Partial_Identification.ipynb)
implements the idea.

## Exercise

Bin the event time into quarters instead of months and see whether the plot
becomes readable.

In [ ]:
# Fill in the blank, then run.
RUN = None          # try True

if RUN:
    s = d.copy()
    s["q"] = np.clip(np.floor(s["k"] / 3), -3, 3).astype(int)
    s.loc[~s["trm"], "q"] = -99
    qs = [q for q in range(-3, 4) if q != -1]
    tq = " + ".join([f"I(trm&(q=={q}))" for q in qs])
    zq = smf.glm("n_uof ~ C(agency_id)+C(year_month)+" + tq, s,
                 family=sm.families.Poisson(), offset=s["lo"]).fit()
    rows = []
    for q in qs:
        k = [c for c in zq.params.index if f"q == {q}" in c][0]
        lo, hi = zq.conf_int().loc[k]
        rows.append({"quarters since start": q,
                     "estimate": f"{pct(zq.params[k]):+.1f}%",
                     "width": round(pct(hi) - pct(lo), 1)})
    display(pd.DataFrame(rows).set_index("quarters since start"))
else:
    print("Set RUN above, then run this cell again.")

<details>
<summary><b>Solution</b></summary>

```python
RUN = True
```

The intervals narrow substantially, because each coefficient now rests on
three times as many agency months, and the pre period coefficients sit much
closer to zero.

**Binning is the right response to an unreadable event study**, and it has a
cost worth stating: a level shift that occurs partway through a bin is
averaged with the months before it, so a sharp response looks gradual.

The general rule is to choose the bin width from the number of units and
months available rather than from the calendar, and to say what was chosen.
Monthly bins on eleven agencies produce a plot that cannot support any
reading; quarterly bins on the same data produce one that can.

</details>

---

**Next:** [Module 7: Staggered Adoption and the Negative Weights Problem](Module_07_Staggered_Adoption.ipynb).

*Part of the Public Safety Statistics Tutorials, developed for the Washington
Data Exchange for Public Safety (WADEPS) through CISER at Washington State
University. Questions or corrections: yin.zhang@wsu.edu*